# Demo: Reputational Risk Classification for New Articles

This notebook demonstrates how to classify new articles (not in the original dataset) using the trained reputational risk classification pipeline.

**Pipeline Overview:**
1. **Sentence Splitting**: Convert article text into sentences
2. **Embedding Generation**: Create semantic embeddings for sentences
3. **Category Scoring**: Compute similarity scores against keyword dictionary
4. **Disambiguation**: Apply tier hierarchy for ambiguous cases
5. **Confidence Gating**: Articles whose top category score falls below the confidence threshold (τ=0.40) are labeled **"No Event"** (no risk category assigned)

**Classification Parameters** (from Cross-Validation):
- Model: `BAAI/bge-m3`
- k_keywords: 6 (top matching keywords per category)
- k_sentences: 2 (top scoring sentences per category)
- ambiguity_delta: 0.01 (threshold for hierarchy application)
- confidence_threshold (τ): 0.40 (below this, an article is labeled "No Event")

**Risk Categories** (7 Total):
- **Tier 1** (Most Foundational): Governance, Personnel
- **Tier 2** (Operational): Products, IT/Data, Processes
- **Tier 3** (External): Legal, Communication

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
import os
import re
import warnings

warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

/home/le/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Libraries imported successfully
PyTorch version: 2.7.1+cu126
CUDA available: True


In [2]:
# Configuration
# Paths
KEYWORD_DICTIONARY_PATH = 'results/phase i/13_final_dictionary_v4.csv'
KEYWORD_EMBEDDINGS_CACHE = 'embeddings/keyword_embeddings_v4.pt'

# Model
MODEL_NAME = 'BAAI/bge-m3'

# Best CV Parameters
BEST_K_KEYWORDS = 6
BEST_K_SENTENCES = 2
BEST_AMBIGUITY_DELTA = 0.01
CONFIDENCE_THRESHOLD = 0.40  # tau: articles with max category score below this are labeled 'No Event'

# Tier hierarchy for ambiguity resolution
TIER_1 = ['Governance', 'Personnel']
TIER_2 = ['Products', 'IT/Data', 'Processes']
TIER_3 = ['Legal', 'Communication']
CATEGORIES = TIER_1 + TIER_2 + TIER_3

print("✓ Configuration loaded")
print(f"\nCategories: {CATEGORIES}")
print(f"Confidence threshold (No Event cutoff): {CONFIDENCE_THRESHOLD}")

✓ Configuration loaded

Categories: ['Governance', 'Personnel', 'Products', 'IT/Data', 'Processes', 'Legal', 'Communication']
Confidence threshold (No Event cutoff): 0.4


In [3]:
# Load keyword dictionary
print("Loading keyword dictionary...")
keyword_df = pd.read_csv(KEYWORD_DICTIONARY_PATH)
keyword_dictionaries = keyword_df.groupby('category')['keyword'].apply(list).to_dict()

print(f"✓ Loaded {len(keyword_df)} keywords across {len(keyword_dictionaries)} categories")
for cat in CATEGORIES:
    if cat in keyword_dictionaries:
        print(f"  {cat}: {len(keyword_dictionaries[cat])} keywords")

Loading keyword dictionary...
✓ Loaded 1109 keywords across 7 categories
  Governance: 202 keywords
  Personnel: 180 keywords
  Products: 121 keywords
  IT/Data: 151 keywords
  Processes: 100 keywords
  Legal: 183 keywords
  Communication: 172 keywords


In [4]:
# Load embedding model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading embedding model on {device}...")
model = SentenceTransformer(MODEL_NAME, device=device)
print("✓ Model loaded")

# Load or create keyword embeddings
print("\nLoading keyword embeddings...")
if os.path.exists(KEYWORD_EMBEDDINGS_CACHE):
    keyword_embeddings_dict = torch.load(KEYWORD_EMBEDDINGS_CACHE)
    print(f"✓ Loaded cached keyword embeddings for {len(keyword_embeddings_dict)} categories")
else:
    print("  Generating keyword embeddings...")
    keyword_embeddings_dict = {
        cat: model.encode(kws, convert_to_tensor=True)
        for cat, kws in keyword_dictionaries.items()
    }
    torch.save(keyword_embeddings_dict, KEYWORD_EMBEDDINGS_CACHE)
    print(f"✓ Keyword embeddings created and cached")

print("\n✓ Model and embeddings ready for inference")

Loading embedding model on cuda...


✓ Model loaded

Loading keyword embeddings...
✓ Loaded cached keyword embeddings for 7 categories

✓ Model and embeddings ready for inference


## Demo Article 1: CrowdStrike Global Tech Outage

This article describes a major IT incident involving CrowdStrike's software update that caused widespread disruptions across multiple industries.

**Expected Category**: Likely **IT/Data** or **Processes** (Tier 2) due to technology/systems focus, or possibly **Legal** if insurance/liability aspects dominate.

In [5]:
# Input Article
article_title = "Insurers face business interruption claims after global tech outage"
article_text = """Insurers could face a raft of business interruption claims after a worldwide tech outage crippled industries from travel to finance on Friday, insurance industry experts said.
A software update by global cybersecurity firm CrowdStrike (CRWD.O), opens new tab appeared to have triggered systems problems that grounded flights, forced some broadcasters off air and left customers without access to services such as healthcare or banking."""

# Combine title and text
full_article = f"{article_title}. {article_text}"

print("Article loaded:")
print(f"Title: {article_title}")
print(f"Text length: {len(article_text)} characters")
print(f"\nFull text:\n{full_article}")

Article loaded:
Title: Insurers face business interruption claims after global tech outage
Text length: 435 characters

Full text:
Insurers face business interruption claims after global tech outage. Insurers could face a raft of business interruption claims after a worldwide tech outage crippled industries from travel to finance on Friday, insurance industry experts said.
A software update by global cybersecurity firm CrowdStrike (CRWD.O), opens new tab appeared to have triggered systems problems that grounded flights, forced some broadcasters off air and left customers without access to services such as healthcare or banking.


In [6]:
# Preprocess: Split text into sentences
def simple_sentence_split(text):
    """Simple sentence splitter using regex"""
    # Split on periods, exclamation marks, question marks followed by space/newline
    sentences = re.split(r'(?<=[.!?])\s+', text)
    # Filter out very short sentences (likely noise)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    return sentences

sentences = simple_sentence_split(full_article)

print(f"✓ Extracted {len(sentences)} sentences")
print("\nSentences:")
for i, sent in enumerate(sentences, 1):
    print(f"  {i}. {sent}")

✓ Extracted 3 sentences

Sentences:
  1. Insurers face business interruption claims after global tech outage.
  2. Insurers could face a raft of business interruption claims after a worldwide tech outage crippled industries from travel to finance on Friday, insurance industry experts said.
  3. A software update by global cybersecurity firm CrowdStrike (CRWD.O), opens new tab appeared to have triggered systems problems that grounded flights, forced some broadcasters off air and left customers without access to services such as healthcare or banking.


In [7]:
# Generate sentence embeddings
print("Generating sentence embeddings...")
sentence_embeddings = model.encode(sentences, convert_to_tensor=True, show_progress_bar=False)
sentence_embeddings_normalized = util.normalize_embeddings(sentence_embeddings)

print(f"✓ Generated embeddings: {sentence_embeddings.shape}")
print(f"  Dimension: {sentence_embeddings.shape[1]}")

Generating sentence embeddings...


✓ Generated embeddings: torch.Size([3, 1024])
  Dimension: 1024


In [8]:
# Compute sentence-level similarity scores for all categories
print(f"Computing category scores (k_keywords={BEST_K_KEYWORDS})...")

sentence_scores_dict = {}
keyword_matches_dict = {}  # Track which keywords matched

for category in CATEGORIES:
    keyword_embeds = keyword_embeddings_dict[category]
    keyword_embeds_normalized = util.normalize_embeddings(keyword_embeds)
    category_keywords = keyword_dictionaries[category]
    
    # Semantic search with top-k keywords
    search_results = util.semantic_search(
        sentence_embeddings_normalized, 
        keyword_embeds_normalized, 
        top_k=min(BEST_K_KEYWORDS, len(keyword_embeds)), 
        score_function=util.dot_score
    )
    
    # Store scores and matched keywords per sentence
    category_scores = []
    category_keyword_matches = []
    
    for hits in search_results:
        # Get scores
        positive_hits = [hit for hit in hits if hit['score'] > 0]
        if positive_hits:
            score = sum(hit['score'] for hit in positive_hits) / len(positive_hits)
            # Store matched keywords
            matched_keywords = [(category_keywords[hit['corpus_id']], hit['score']) 
                              for hit in positive_hits]
            category_keyword_matches.append(matched_keywords)
        else:
            score = 0.0
            category_keyword_matches.append([])
        
        category_scores.append(score)
    
    sentence_scores_dict[category] = category_scores
    keyword_matches_dict[category] = category_keyword_matches

# Create DataFrame with sentence scores
sentence_scores_df = pd.DataFrame(sentence_scores_dict)
sentence_scores_df['sentence'] = sentences

print("✓ Sentence-level scores computed")
print(f"\nSentence Scores Preview:")
print(sentence_scores_df.head())

Computing category scores (k_keywords=6)...
✓ Sentence-level scores computed

Sentence Scores Preview:
   Governance  Personnel  Products   IT/Data  Processes     Legal  \
0    0.470701   0.523659  0.502174  0.551026   0.600393  0.509944   
1    0.402716   0.442202  0.440891  0.470824   0.526816  0.402965   
2    0.471492   0.500905  0.487430  0.551569   0.568778  0.451893   

   Communication                                           sentence  
0       0.482746  Insurers face business interruption claims aft...  
1       0.413400  Insurers could face a raft of business interru...  
2       0.484510  A software update by global cybersecurity firm...  


In [9]:
# Aggregate to article level (top k_sentences per category)
print(f"Aggregating to article level (k_sentences={BEST_K_SENTENCES})...")

k_use = min(BEST_K_SENTENCES, len(sentences))

article_scores = {
    cat: sentence_scores_df.nlargest(k_use, cat)[cat].mean() 
    for cat in CATEGORIES
}

print(f"✓ Article-level scores computed using top {k_use} sentences per category")
print(f"\nArticle Scores:")
for cat, score in sorted(article_scores.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cat:15s}: {score:.4f}")

Aggregating to article level (k_sentences=2)...
✓ Article-level scores computed using top 2 sentences per category

Article Scores:
  Processes      : 0.5846
  IT/Data        : 0.5513
  Personnel      : 0.5123
  Products       : 0.4948
  Communication  : 0.4836
  Legal          : 0.4809
  Governance     : 0.4711


In [10]:
# Classification with ambiguity resolution
print("="*80)
print("CLASSIFICATION RESULT")
print("="*80)

# Sort scores in descending order
sorted_scores = sorted(article_scores.items(), key=lambda x: x[1], reverse=True)

top_category = sorted_scores[0][0]
top_score = sorted_scores[0][1]
second_category = sorted_scores[1][0]
second_score = sorted_scores[1][1]

score_delta = top_score - second_score

# Check if ambiguous (within ambiguity_delta)
is_ambiguous = score_delta < BEST_AMBIGUITY_DELTA

if is_ambiguous:
    print(f"\n⚠️  AMBIGUOUS CASE (score_delta={score_delta:.6f} < {BEST_AMBIGUITY_DELTA})")

    # Collect all categories within ambiguity_delta of the top score
    ambiguous_categories = [cat for cat, score in sorted_scores 
                           if (top_score - score) < BEST_AMBIGUITY_DELTA]

    print(f"   Competing categories: {', '.join(ambiguous_categories)}")
    print(f"\n   Applying tier hierarchy...")

    # Apply hierarchy
    predicted_category = None

    # Check Tier 1 first
    tier_1_set = set(TIER_1)
    for cat in TIER_1:
        if cat in ambiguous_categories:
            predicted_category = cat
            print(f"   → Selected {cat} (Tier 1: Most Foundational)")
            break

    # If no Tier 1, check Tier 2
    if not predicted_category:
        for cat in TIER_2:
            if cat in ambiguous_categories:
                predicted_category = cat
                print(f"   → Selected {cat} (Tier 2: Operational)")
                break

    # If still none, use Tier 3 or top scoring
    if not predicted_category:
        for cat in TIER_3:
            if cat in ambiguous_categories:
                predicted_category = cat
                print(f"   → Selected {cat} (Tier 3: External)")
                break

    if not predicted_category:
        predicted_category = top_category
        print(f"   → Selected {top_category} (Highest Score)")
else:
    print(f"\n✓ CLEAR WINNER (score_delta={score_delta:.6f} >= {BEST_AMBIGUITY_DELTA})")
    predicted_category = top_category

# Apply confidence threshold: override to 'No Event' if below tau
# (matches production pipeline: tier resolution runs first, then the
# threshold gate overrides low-confidence articles regardless of tier)
below_threshold = top_score < CONFIDENCE_THRESHOLD
if below_threshold:
    print(f"\n⚠️  BELOW CONFIDENCE THRESHOLD (top_score={top_score:.4f} < {CONFIDENCE_THRESHOLD}) "
          f"→ overriding category to 'No Event'")
    predicted_category = 'No Event'

print(f"\n{'='*80}")
print(f"PREDICTED CATEGORY: {predicted_category}")
print(f"{'='*80}")
print(f"\nTop Score: {top_score:.4f}")
print(f"Second Best: {second_category} ({second_score:.4f})")
print(f"Score Delta: {score_delta:.6f}")
print(f"Ambiguous: {'Yes' if is_ambiguous else 'No'}")
print(f"Below Confidence Threshold (No Event): {'Yes' if below_threshold else 'No'}")


CLASSIFICATION RESULT

✓ CLEAR WINNER (score_delta=0.033288 >= 0.01)

PREDICTED CATEGORY: Processes

Top Score: 0.5846
Second Best: IT/Data (0.5513)
Score Delta: 0.033288
Ambiguous: No
Below Confidence Threshold (No Event): No


In [11]:
# Detailed Score Breakdown
print("\n" + "="*80)
print("DETAILED SCORE BREAKDOWN")
print("="*80)

# Create a detailed results DataFrame
results_df = pd.DataFrame([
    {
        'Category': cat,
        'Score': score,
        'Tier': 'Tier 1' if cat in TIER_1 else ('Tier 2' if cat in TIER_2 else 'Tier 3'),
        'Rank': rank + 1,
        'Delta from Top': top_score - score
    }
    for rank, (cat, score) in enumerate(sorted_scores)
])

print("\n" + results_df.to_string(index=False))

# Show top scoring sentences for the highest-scoring category (used as
# reference even when the article was gated to 'No Event', since that
# label has no corresponding score column)
display_category = predicted_category if predicted_category != 'No Event' else top_category
print(f"\n{'='*80}")
header_suffix = " (highest-scoring category; article labeled 'No Event')" if predicted_category == 'No Event' else ""
print(f"TOP SCORING SENTENCES FOR '{display_category}'{header_suffix}")
print(f"{'='*80}")

top_sentences = sentence_scores_df.nlargest(k_use, display_category)
for idx, row in top_sentences.iterrows():
    print(f"\nScore: {row[display_category]:.4f}")
    print(f"  → {row['sentence']}")



DETAILED SCORE BREAKDOWN

     Category    Score   Tier  Rank  Delta from Top
    Processes 0.584586 Tier 2     1        0.000000
      IT/Data 0.551298 Tier 2     2        0.033288
    Personnel 0.512282 Tier 1     3        0.072304
     Products 0.494802 Tier 2     4        0.089783
Communication 0.483628 Tier 3     5        0.100957
        Legal 0.480919 Tier 3     6        0.103667
   Governance 0.471096 Tier 1     7        0.113489

TOP SCORING SENTENCES FOR 'Processes'

Score: 0.6004
  → Insurers face business interruption claims after global tech outage.

Score: 0.5688
  → A software update by global cybersecurity firm CrowdStrike (CRWD.O), opens new tab appeared to have triggered systems problems that grounded flights, forced some broadcasters off air and left customers without access to services such as healthcare or banking.


In [12]:
# Show top matching keywords for the highest-scoring category
display_category = predicted_category if predicted_category != 'No Event' else top_category
print(f"\n{'='*80}")
print(f"TOP MATCHING KEYWORDS FOR '{display_category}'")
print(f"{'='*80}")

# Collect all keyword matches across sentences for the display category
all_keyword_scores = []
category_matches = keyword_matches_dict[display_category]

for sent_idx, matches in enumerate(category_matches):
    for keyword, score in matches:
        all_keyword_scores.append({
            'keyword': keyword,
            'score': score,
            'sentence_idx': sent_idx,
            'sentence': sentences[sent_idx]
        })

# Sort by score and get top keywords
if all_keyword_scores:
    keyword_df = pd.DataFrame(all_keyword_scores)

    # Show unique top keywords
    top_keywords = keyword_df.groupby('keyword')['score'].max().sort_values(ascending=False).head(10)

    print(f"\nTop 10 Matched Keywords (by similarity score):")
    for idx, (keyword, score) in enumerate(top_keywords.items(), 1):
        # Find which sentence this keyword matched
        best_match = keyword_df[keyword_df['keyword'] == keyword].nlargest(1, 'score').iloc[0]
        print(f"\n  {idx}. '{keyword}' (score: {score:.4f})")
        print(f"     Matched in: \"{best_match['sentence'][:80]}...\"")
else:
    print("\nNo keyword matches found.")



TOP MATCHING KEYWORDS FOR 'Processes'

Top 10 Matched Keywords (by similarity score):

  1. 'System interruption' (score: 0.6504)
     Matched in: "Insurers face business interruption claims after global tech outage...."

  2. 'Service delivery interruption' (score: 0.6501)
     Matched in: "Insurers face business interruption claims after global tech outage...."

  3. 'Supply chain disruption' (score: 0.6268)
     Matched in: "Insurers face business interruption claims after global tech outage...."

  4. 'Operational disruption' (score: 0.6119)
     Matched in: "Insurers face business interruption claims after global tech outage...."

  5. 'Software update complications' (score: 0.5709)
     Matched in: "A software update by global cybersecurity firm CrowdStrike (CRWD.O), opens new t..."

  6. 'Network maintenance delays' (score: 0.5532)
     Matched in: "A software update by global cybersecurity firm CrowdStrike (CRWD.O), opens new t..."

  7. 'Billing system malfunctions' (score: 0

## Demo Article 2: AstraZeneca Drug Label Warning

This article describes a regulatory/product-safety action: AstraZeneca adding arrhythmia-related warnings to a drug label.

**Expected Category**: Likely **Products** or **Legal**, depending on whether the regulatory or product-safety framing dominates.

In [13]:
# Input Article
article_title = "Heart Warning Added to Label on Popular Antipsychotic Drug"
article_text = """AstraZeneca will add warnings to labels for Seroquel saying that the drug should be avoided in combination with 12 medicines linked to arrhythmia. """

# Combine title and text
full_article = f"{article_title}. {article_text}"

print("Article loaded:")
print(f"Title: {article_title}")
print(f"Text length: {len(article_text)} characters")
print(f"\nFull text:\n{full_article}")

Article loaded:
Title: Heart Warning Added to Label on Popular Antipsychotic Drug
Text length: 147 characters

Full text:
Heart Warning Added to Label on Popular Antipsychotic Drug. AstraZeneca will add warnings to labels for Seroquel saying that the drug should be avoided in combination with 12 medicines linked to arrhythmia. 


In [14]:
# Preprocess: Split text into sentences
def simple_sentence_split(text):
    """Simple sentence splitter using regex"""
    # Split on periods, exclamation marks, question marks followed by space/newline
    sentences = re.split(r'(?<=[.!?])\s+', text)
    # Filter out very short sentences (likely noise)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    return sentences

sentences = simple_sentence_split(full_article)

print(f"✓ Extracted {len(sentences)} sentences")
print("\nSentences:")
for i, sent in enumerate(sentences, 1):
    print(f"  {i}. {sent}")

✓ Extracted 2 sentences

Sentences:
  1. Heart Warning Added to Label on Popular Antipsychotic Drug.
  2. AstraZeneca will add warnings to labels for Seroquel saying that the drug should be avoided in combination with 12 medicines linked to arrhythmia.


In [15]:
# Generate sentence embeddings
print("Generating sentence embeddings...")
sentence_embeddings = model.encode(sentences, convert_to_tensor=True, show_progress_bar=False)
sentence_embeddings_normalized = util.normalize_embeddings(sentence_embeddings)

print(f"✓ Generated embeddings: {sentence_embeddings.shape}")
print(f"  Dimension: {sentence_embeddings.shape[1]}")

Generating sentence embeddings...
✓ Generated embeddings: torch.Size([2, 1024])
  Dimension: 1024


In [16]:
# Compute sentence-level similarity scores for all categories
print(f"Computing category scores (k_keywords={BEST_K_KEYWORDS})...")

sentence_scores_dict = {}
keyword_matches_dict = {}  # Track which keywords matched

for category in CATEGORIES:
    keyword_embeds = keyword_embeddings_dict[category]
    keyword_embeds_normalized = util.normalize_embeddings(keyword_embeds)
    category_keywords = keyword_dictionaries[category]
    
    # Semantic search with top-k keywords
    search_results = util.semantic_search(
        sentence_embeddings_normalized, 
        keyword_embeds_normalized, 
        top_k=min(BEST_K_KEYWORDS, len(keyword_embeds)), 
        score_function=util.dot_score
    )
    
    # Store scores and matched keywords per sentence
    category_scores = []
    category_keyword_matches = []
    
    for hits in search_results:
        # Get scores
        positive_hits = [hit for hit in hits if hit['score'] > 0]
        if positive_hits:
            score = sum(hit['score'] for hit in positive_hits) / len(positive_hits)
            # Store matched keywords
            matched_keywords = [(category_keywords[hit['corpus_id']], hit['score']) 
                              for hit in positive_hits]
            category_keyword_matches.append(matched_keywords)
        else:
            score = 0.0
            category_keyword_matches.append([])
        
        category_scores.append(score)
    
    sentence_scores_dict[category] = category_scores
    keyword_matches_dict[category] = category_keyword_matches

# Create DataFrame with sentence scores
sentence_scores_df = pd.DataFrame(sentence_scores_dict)
sentence_scores_df['sentence'] = sentences

print("✓ Sentence-level scores computed")
print(f"\nSentence Scores Preview:")
print(sentence_scores_df.head())

Computing category scores (k_keywords=6)...
✓ Sentence-level scores computed

Sentence Scores Preview:
   Governance  Personnel  Products   IT/Data  Processes     Legal  \
0    0.448148   0.457167  0.531675  0.470226   0.424184  0.471607   
1    0.400306   0.377867  0.440412  0.383753   0.404482  0.407529   

   Communication                                           sentence  
0       0.517198  Heart Warning Added to Label on Popular Antips...  
1       0.439714  AstraZeneca will add warnings to labels for Se...  


In [17]:
# Aggregate to article level (top k_sentences per category)
print(f"Aggregating to article level (k_sentences={BEST_K_SENTENCES})...")

k_use = min(BEST_K_SENTENCES, len(sentences))

article_scores = {
    cat: sentence_scores_df.nlargest(k_use, cat)[cat].mean() 
    for cat in CATEGORIES
}

print(f"✓ Article-level scores computed using top {k_use} sentences per category")
print(f"\nArticle Scores:")
for cat, score in sorted(article_scores.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cat:15s}: {score:.4f}")

Aggregating to article level (k_sentences=2)...
✓ Article-level scores computed using top 2 sentences per category

Article Scores:
  Products       : 0.4860
  Communication  : 0.4785
  Legal          : 0.4396
  IT/Data        : 0.4270
  Governance     : 0.4242
  Personnel      : 0.4175
  Processes      : 0.4143


In [18]:
# Classification with ambiguity resolution
print("="*80)
print("CLASSIFICATION RESULT")
print("="*80)

# Sort scores in descending order
sorted_scores = sorted(article_scores.items(), key=lambda x: x[1], reverse=True)

top_category = sorted_scores[0][0]
top_score = sorted_scores[0][1]
second_category = sorted_scores[1][0]
second_score = sorted_scores[1][1]

score_delta = top_score - second_score

# Check if ambiguous (within ambiguity_delta)
is_ambiguous = score_delta < BEST_AMBIGUITY_DELTA

if is_ambiguous:
    print(f"\n⚠️  AMBIGUOUS CASE (score_delta={score_delta:.6f} < {BEST_AMBIGUITY_DELTA})")

    # Collect all categories within ambiguity_delta of the top score
    ambiguous_categories = [cat for cat, score in sorted_scores 
                           if (top_score - score) < BEST_AMBIGUITY_DELTA]

    print(f"   Competing categories: {', '.join(ambiguous_categories)}")
    print(f"\n   Applying tier hierarchy...")

    # Apply hierarchy
    predicted_category = None

    # Check Tier 1 first
    tier_1_set = set(TIER_1)
    for cat in TIER_1:
        if cat in ambiguous_categories:
            predicted_category = cat
            print(f"   → Selected {cat} (Tier 1: Most Foundational)")
            break

    # If no Tier 1, check Tier 2
    if not predicted_category:
        for cat in TIER_2:
            if cat in ambiguous_categories:
                predicted_category = cat
                print(f"   → Selected {cat} (Tier 2: Operational)")
                break

    # If still none, use Tier 3 or top scoring
    if not predicted_category:
        for cat in TIER_3:
            if cat in ambiguous_categories:
                predicted_category = cat
                print(f"   → Selected {cat} (Tier 3: External)")
                break

    if not predicted_category:
        predicted_category = top_category
        print(f"   → Selected {top_category} (Highest Score)")
else:
    print(f"\n✓ CLEAR WINNER (score_delta={score_delta:.6f} >= {BEST_AMBIGUITY_DELTA})")
    predicted_category = top_category

# Apply confidence threshold: override to 'No Event' if below tau
# (matches production pipeline: tier resolution runs first, then the
# threshold gate overrides low-confidence articles regardless of tier)
below_threshold = top_score < CONFIDENCE_THRESHOLD
if below_threshold:
    print(f"\n⚠️  BELOW CONFIDENCE THRESHOLD (top_score={top_score:.4f} < {CONFIDENCE_THRESHOLD}) "
          f"→ overriding category to 'No Event'")
    predicted_category = 'No Event'

print(f"\n{'='*80}")
print(f"PREDICTED CATEGORY: {predicted_category}")
print(f"{'='*80}")
print(f"\nTop Score: {top_score:.4f}")
print(f"Second Best: {second_category} ({second_score:.4f})")
print(f"Score Delta: {score_delta:.6f}")
print(f"Ambiguous: {'Yes' if is_ambiguous else 'No'}")
print(f"Below Confidence Threshold (No Event): {'Yes' if below_threshold else 'No'}")


CLASSIFICATION RESULT

⚠️  AMBIGUOUS CASE (score_delta=0.007587 < 0.01)
   Competing categories: Products, Communication

   Applying tier hierarchy...
   → Selected Products (Tier 2: Operational)

PREDICTED CATEGORY: Products

Top Score: 0.4860
Second Best: Communication (0.4785)
Score Delta: 0.007587
Ambiguous: Yes
Below Confidence Threshold (No Event): No


In [19]:
# Detailed Score Breakdown
print("\n" + "="*80)
print("DETAILED SCORE BREAKDOWN")
print("="*80)

# Create a detailed results DataFrame
results_df = pd.DataFrame([
    {
        'Category': cat,
        'Score': score,
        'Tier': 'Tier 1' if cat in TIER_1 else ('Tier 2' if cat in TIER_2 else 'Tier 3'),
        'Rank': rank + 1,
        'Delta from Top': top_score - score
    }
    for rank, (cat, score) in enumerate(sorted_scores)
])

print("\n" + results_df.to_string(index=False))

# Show top scoring sentences for the highest-scoring category (used as
# reference even when the article was gated to 'No Event', since that
# label has no corresponding score column)
display_category = predicted_category if predicted_category != 'No Event' else top_category
print(f"\n{'='*80}")
header_suffix = " (highest-scoring category; article labeled 'No Event')" if predicted_category == 'No Event' else ""
print(f"TOP SCORING SENTENCES FOR '{display_category}'{header_suffix}")
print(f"{'='*80}")

top_sentences = sentence_scores_df.nlargest(k_use, display_category)
for idx, row in top_sentences.iterrows():
    print(f"\nScore: {row[display_category]:.4f}")
    print(f"  → {row['sentence']}")



DETAILED SCORE BREAKDOWN

     Category    Score   Tier  Rank  Delta from Top
     Products 0.486043 Tier 2     1        0.000000
Communication 0.478456 Tier 3     2        0.007587
        Legal 0.439568 Tier 3     3        0.046475
      IT/Data 0.426990 Tier 2     4        0.059054
   Governance 0.424227 Tier 1     5        0.061816
    Personnel 0.417517 Tier 1     6        0.068526
    Processes 0.414333 Tier 2     7        0.071710

TOP SCORING SENTENCES FOR 'Products'

Score: 0.5317
  → Heart Warning Added to Label on Popular Antipsychotic Drug.

Score: 0.4404
  → AstraZeneca will add warnings to labels for Seroquel saying that the drug should be avoided in combination with 12 medicines linked to arrhythmia.


In [20]:
# Show top matching keywords for the highest-scoring category
display_category = predicted_category if predicted_category != 'No Event' else top_category
print(f"\n{'='*80}")
print(f"TOP MATCHING KEYWORDS FOR '{display_category}'")
print(f"{'='*80}")

# Collect all keyword matches across sentences for the display category
all_keyword_scores = []
category_matches = keyword_matches_dict[display_category]

for sent_idx, matches in enumerate(category_matches):
    for keyword, score in matches:
        all_keyword_scores.append({
            'keyword': keyword,
            'score': score,
            'sentence_idx': sent_idx,
            'sentence': sentences[sent_idx]
        })

# Sort by score and get top keywords
if all_keyword_scores:
    keyword_df = pd.DataFrame(all_keyword_scores)

    # Show unique top keywords
    top_keywords = keyword_df.groupby('keyword')['score'].max().sort_values(ascending=False).head(10)

    print(f"\nTop 10 Matched Keywords (by similarity score):")
    for idx, (keyword, score) in enumerate(top_keywords.items(), 1):
        # Find which sentence this keyword matched
        best_match = keyword_df[keyword_df['keyword'] == keyword].nlargest(1, 'score').iloc[0]
        print(f"\n  {idx}. '{keyword}' (score: {score:.4f})")
        print(f"     Matched in: \"{best_match['sentence'][:80]}...\"")
else:
    print("\nNo keyword matches found.")



TOP MATCHING KEYWORDS FOR 'Products'

Top 10 Matched Keywords (by similarity score):

  1. 'Toxic substance in product' (score: 0.5470)
     Matched in: "Heart Warning Added to Label on Popular Antipsychotic Drug...."

  2. 'Harmful substance detected' (score: 0.5457)
     Matched in: "Heart Warning Added to Label on Popular Antipsychotic Drug...."

  3. 'Choking hazard product' (score: 0.5267)
     Matched in: "Heart Warning Added to Label on Popular Antipsychotic Drug...."

  4. 'Product ingredient mislabeling' (score: 0.5243)
     Matched in: "Heart Warning Added to Label on Popular Antipsychotic Drug...."

  5. 'Incorrect product labeling' (score: 0.5232)
     Matched in: "Heart Warning Added to Label on Popular Antipsychotic Drug...."

  6. 'Incorrect product dosage' (score: 0.5231)
     Matched in: "Heart Warning Added to Label on Popular Antipsychotic Drug...."


## How to Classify Your Own Articles

To classify additional articles from your paper:

1. **Add or modify an input cell** (see Cell 6 or Cell 15 for examples) with your article title and text:
   ```python
   article_title = "Your article title here"
   article_text = """Your article text here"""
   ```

2. **Run the input cell and every processing cell after it** (sentence splitting through the keyword breakdown), or simply **Run All**

3. **Results will show:**
   - Predicted category with confidence scores, or **"No Event"** if the top score falls below the confidence threshold (τ=0.40)
   - Whether ambiguity resolution was applied
   - Top scoring sentences that influenced the decision
   - Complete score breakdown across all 7 categories

**Tips:**
- Longer articles may take a few seconds to process
- The pipeline automatically handles sentence splitting
- Ambiguous cases (score_delta < 0.01) trigger tier hierarchy
- Articles with max category score below τ=0.40 are labeled "No Event" (no risk category assigned), overriding any tier-based resolution
- Check the "Top Scoring Sentences" to understand why a category was selected